# Notebook 01: Intake and Normalization
## Goal: Raw Agency Notes $\\rightarrow$ CanonicalPacket v0.1

This notebook prototypes the "First Mile" of the compiler: taking unstructured or semi-structured intake and producing a high-integrity state object with provenance and authority levels.

**Key Design Principles:**
1. Every fact has provenance - no fact without evidence
2. Authority levels are explicit and ordered
3. Unknowns are explicit, not just absent
4. Facts, derived signals, and hypotheses are separated
5. Event-log ready - all mutations go through controlled functions

In [ ]:
import json
from typing import List, Dict, Any, Optional, Literal, Tuple
from dataclasses import dataclass, asdict, field
from datetime import datetime
from enum import Enum, IntEnum
import uuid

# =============================================================================
# SECTION 1: CORE DOMAIN MODELS
# =============================================================================

class AuthorityLevel(IntEnum):
    """
    Ordered authority levels - lower number = higher authority.
    
    CORRECT ORDER (per spec):
    manual_override > explicit_user > imported_structured > explicit_owner > 
    derived_signal > soft_hypothesis > unknown
    """
    MANUAL_OVERRIDE = 1       # Highest - human override
    EXPLICIT_USER = 2         # Direct user input
    IMPORTED_STRUCTURED = 3   # CRM/form/system import
    EXPLICIT_OWNER = 4        # Agency owner's notes
    DERIVED_SIGNAL = 5        # Computed from facts
    SOFT_HYPOTHESIS = 6       # Weak inference
    UNKNOWN = 7               # Lowest - no authority

# Authority comparison helper
def higher_authority(auth1: str, auth2: str) -> str:
    """Return the higher authority (lower number = higher)."""
    try:
        a1 = AuthorityLevel[auth1.upper()] if isinstance(auth1, str) else auth1
        a2 = AuthorityLevel[auth2.upper()] if isinstance(auth2, str) else auth2
        return auth1 if a1.value < a2.value else auth2
    except (KeyError, ValueError):
        return auth1

def is_hypothesis(authority_level: str) -> bool:
    """Check if this authority level represents a hypothesis (not a fact)."""
    return authority_level.lower() in ("soft_hypothesis", "derived_signal")

def is_fact(authority_level: str) -> bool:
    """Check if this authority level represents an actual fact."""
    return authority_level.lower() in ("manual_override", "explicit_user", 
                                        "imported_structured", "explicit_owner")

@dataclass
class EvidenceRef:
    """Provenance reference - links a value to its source"""
    ref_id: str
    envelope_id: str  # which source envelope this evidence came from
    evidence_type: Literal["text_span", "structured_field", "attachment_extract", "derived", "manual_entry"]
    excerpt: str  # the actual text or field path
    field_path: Optional[str] = None  # for structured fields: e.g., "traveler_preferences.budget"
    offset: Optional[Dict[str, int]] = None  # for text: {"start": 42, "end": 51}
    confidence: float = 1.0
    metadata: Dict[str, Any] = field(default_factory=dict)

class ExtractionMode:
    """Valid extraction modes - semantic meaning matters for MVB validation."""
    DIRECT_EXTRACT = "direct_extract"    # Verbatim from source
    IMPORTED = "imported"                 # From structured import
    NORMALIZED = "normalized"             # Value standardized (e.g., blr -> Bangalore)
    DERIVED = "derived"                   # Computed from facts
    MANUAL_ENTRY = "manual_entry"         # Human entered

@dataclass
class Slot:
    """A field container with full provenance and authority tracking"""
    value: Any = None
    confidence: float = 0.0
    authority_level: str = "unknown"
    extraction_mode: str = "unknown"  # Use ExtractionMode constants
    evidence_refs: List[EvidenceRef] = field(default_factory=list)
    updated_at: str = field(default_factory=lambda: datetime.now().isoformat())
    notes: Optional[str] = None

@dataclass
class UnknownField:
    """Explicit representation of unknown/missing fields"""
    field_name: str
    reason: Literal["not_present_in_source", "not_extracted_yet", "extraction_failed", "intentionally_unknown"]
    attempted_at: Optional[str] = None
    notes: Optional[str] = None

# =============================================================================
# SECTION 2: SOURCE ENVELOPE
# =============================================================================

@dataclass
class SourceEnvelope:
    """
    The canonical input wrapper - preserves all source information
    before normalization and extraction.
    """
    envelope_id: str
    source_system: Literal["agency_notes", "traveler_form", "email_thread", "chat_history", "structured_import"]
    actor_type: Literal["agent", "traveler", "owner", "system"]
    received_at: str
    content: Any  # raw content (string or dict)
    content_type: Literal["freeform_text", "structured_json", "hybrid"]
    attachments: List[Dict[str, Any]] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    @classmethod
    def from_freeform(cls, text: str, source: str = "agency_notes", actor: str = "agent") -> "SourceEnvelope":
        """Factory for freeform text input"""
        return cls(
            envelope_id=f"env_{uuid.uuid4().hex[:8]}",
            source_system=source,
            actor_type=actor,
            received_at=datetime.now().isoformat(),
            content=text,
            content_type="freeform_text"
        )
    
    @classmethod
    def from_structured(cls, data: Dict, source: str = "structured_import", actor: str = "system") -> "SourceEnvelope":
        """Factory for structured input"""
        return cls(
            envelope_id=f"env_{uuid.uuid4().hex[:8]}",
            source_system=source,
            actor_type=actor,
            received_at=datetime.now().isoformat(),
            content=data,
            content_type="structured_json"
        )

# =============================================================================
# SECTION 3: CANONICAL PACKET (STATE OBJECT)
# =============================================================================

@dataclass
class CanonicalPacket:
    """
    The single source of truth for all extracted information.
    This is the state object that flows through the compiler stages.
    
    CRITICAL: Facts, derived signals, and hypotheses are STRICTLY SEPARATED.
    - facts: Only explicit extracted/imported data (authority: manual_override, explicit_user, 
             imported_structured, explicit_owner)
    - derived_signals: Computed from facts (authority: derived_signal)
    - hypotheses: Weak inferences (authority: soft_hypothesis)
    """
    packet_id: str
    created_at: str
    last_updated: str
    
    # Core data layers - STRICTLY SEPARATED
    facts: Dict[str, Slot] = field(default_factory=dict)           # Explicit extracted facts ONLY
    derived_signals: Dict[str, Slot] = field(default_factory=dict)  # Computed from facts
    hypotheses: Dict[str, Slot] = field(default_factory=dict)       # Soft hypotheses
    
    # Explicit tracking
    unknowns: List[UnknownField] = field(default_factory=list)       # Explicit unknown fields
    contradictions: List[Dict[str, Any]] = field(default_factory=list)  # Conflicting values
    source_envelope_ids: List[str] = field(default_factory=list)    # All source envelopes
    
    # Stage and decision
    stage: str = "discovery"
    decision_state: Optional[str] = None
    
    # Audit trail
    event_cursor: int = 0
    events: List[Dict[str, Any]] = field(default_factory=list)      # Append-only event log
    
    def _emit_event(self, event_type: str, details: Dict[str, Any]) -> None:
        """All state mutations must go through this - event log ready"""
        self.event_cursor += 1
        self.events.append({
            "event_id": self.event_cursor,
            "event_type": event_type,
            "timestamp": datetime.now().isoformat(),
            "details": details
        })
        self.last_updated = datetime.now().isoformat()
    
    def set_fact(self, field_name: str, slot: Slot) -> None:
        """
        Set a fact with event emission.
        CRITICAL: Only accepts slots with fact-level authority.
        """
        if not is_fact(slot.authority_level):
            raise ValueError(f"Cannot set_fact with authority_level='{slot.authority_level}'. "
                           f"Facts require manual_override, explicit_user, imported_structured, or explicit_owner.")
        self.facts[field_name] = slot
        self._emit_event("fact_set", {
            "field_name": field_name,
            "value": slot.value,
            "authority_level": slot.authority_level
        })
    
    def set_derived_signal(self, field_name: str, slot: Slot) -> None:
        """
        Set a derived signal. Must be derived from facts, with derivation chain.
        CRITICAL: Only accepts derived_signal authority.
        """
        if slot.authority_level.lower() != "derived_signal":
            raise ValueError(f"Cannot set_derived_signal with authority_level='{slot.authority_level}'. "
                           f"Derived signals require authority_level='derived_signal'.")
        self.derived_signals[field_name] = slot
        self._emit_event("derived_signal_set", {
            "field_name": field_name,
            "value": slot.value,
            "derived_from": [e.excerpt for e in slot.evidence_refs]
        })
    
    def set_hypothesis(self, field_name: str, slot: Slot) -> None:
        """
        Set a soft hypothesis. Cannot override facts.
        CRITICAL: Only accepts soft_hypothesis authority.
        """
        if slot.authority_level.lower() != "soft_hypothesis":
            raise ValueError(f"Cannot set_hypothesis with authority_level='{slot.authority_level}'. "
                           f"Hypotheses require authority_level='soft_hypothesis'.")
        self.hypotheses[field_name] = slot
        self._emit_event("hypothesis_set", {
            "field_name": field_name,
            "value": slot.value,
            "confidence": slot.confidence
        })
    
    def add_unknown(self, field_name: str, reason: str) -> None:
        """Explicitly mark a field as unknown"""
        self.unknowns.append(UnknownField(
            field_name=field_name,
            reason=reason
        ))
        self._emit_event("unknown_added", {"field_name": field_name, "reason": reason})
    
    def add_contradiction(self, field_name: str, values: List[Any], sources: List[str]) -> None:
        """Record a contradiction between sources"""
        self.contradictions.append({
            "field_name": field_name,
            "values": values,
            "sources": sources,
            "detected_at": datetime.now().isoformat()
        })
        self._emit_event("contradiction_detected", {
            "field_name": field_name,
            "values": values
        })

# =============================================================================
# SECTION 4: CONFLICT RESOLUTION POLICY
# =============================================================================

class ConflictResolutionPolicy:
    """
    Named, testable, configurable conflict resolution policy.
    Determines how to resolve conflicts between different sources.
    """
    
    @staticmethod
    def resolve(field_name: str, 
                freeform_value: Any, freeform_authority: str, freeform_confidence: float,
                structured_value: Any, structured_authority: str, structured_confidence: float) -> Dict[str, Any]:
        """
        Resolve conflict between freeform and structured values.
        
        Returns dict with:
        - resolved_value: The chosen value
        - authority: The authority of the chosen value
        - resolution_reason: Why this value was chosen
        """
        # Policy: structured > freeform when authorities are equal
        # But explicit_user > imported_structured
        
        auth_order = {
            "manual_override": 1,
            "explicit_user": 2,
            "imported_structured": 3,
            "explicit_owner": 4,
            "derived_signal": 5,
            "soft_hypothesis": 6,
            "unknown": 7
        }
        
        freeform_rank = auth_order.get(freeform_authority.lower(), 7)
        structured_rank = auth_order.get(structured_authority.lower(), 7)
        
        # If freeform has higher authority (lower rank), use it
        if freeform_rank < structured_rank:
            return {
                "resolved_value": freeform_value,
                "authority": freeform_authority,
                "resolution_reason": f"Freeform has higher authority ({freeform_authority} > {structured_authority})",
                "extraction_mode": "direct_extract"
            }
        
        # If structured has higher or equal authority, use it
        if structured_rank <= freeform_rank:
            return {
                "resolved_value": structured_value,
                "authority": structured_authority,
                "resolution_reason": f"Structured has equal or higher authority ({structured_authority})",
                "extraction_mode": "imported"
            }
        
        # Fallback to structured (conservative)
        return {
            "resolved_value": structured_value,
            "authority": structured_authority,
            "resolution_reason": "Default policy: prefer structured when unclear",
            "extraction_mode": "imported"
        }

# =============================================================================
# SECTION 5: SOURCE ADAPTER
# =============================================================================

class SourceAdapter:
    """
    Transforms raw inputs into SourceEnvelopes.
    Handles freeform, structured, and hybrid inputs.
    """
    
    @staticmethod
    def adapt_freeform(text: str, source: str = "agency_notes", actor: str = "agent") -> SourceEnvelope:
        """Adapt freeform text into envelope"""
        return SourceEnvelope.from_freeform(text, source, actor)
    
    @staticmethod
    def adapt_structured(data: Dict, source: str = "structured_import", actor: str = "system") -> SourceEnvelope:
        """Adapt structured data into envelope"""
        return SourceEnvelope.from_structured(data, source, actor)
    
    @staticmethod
    def adapt_hybrid(text: str, data: Dict, source: str = "agency_notes", actor: str = "agent") -> SourceEnvelope:
        """Adapt hybrid input - combines text and structured data"""
        return SourceEnvelope(
            envelope_id=f"env_{uuid.uuid4().hex[:8]}",
            source_system=source,
            actor_type=actor,
            received_at=datetime.now().isoformat(),
            content={"text": text, "structured": data},
            content_type="hybrid",
            metadata={"hybrid_sources": ["text", "structured"]}
        )

# =============================================================================
# SECTION 6: NORMALIZATION UTILITIES
# =============================================================================

class Normalizer:
    """
    Standardizes values - does NOT speculate.
    Only transforms known mappings, never infers.
    """
    
    CITY_NORMALIZATIONS = {
        "blr": "Bangalore",
        "bengaluru": "Bangalore",
        "nyc": "New York",
        "del": "Delhi",
        "deli": "Delhi",
        "nrt": "Narita",
        "hnd": "Haneda",
        "lax": "Los Angeles",
        "sfo": "San Francisco"
    }
    
    BUDGET_NORMALIZATIONS = {
        "mid-range": "mid_range",
        "mid range": "mid_range",
        "budget": "budget",
        "economy": "budget",
        "luxury": "luxury",
        "premium": "luxury",
        "high-end": "luxury"
    }
    
    @classmethod
    def normalize_city(cls, raw: str) -> Tuple[str, bool]:
        """
        Normalize city names.
        Returns (normalized_value, was_normalized).
        """
        normalized = cls.CITY_NORMALIZATIONS.get(raw.lower(), raw)
        was_normalized = normalized.lower() != raw.lower()
        return normalized, was_normalized
    
    @classmethod
    def normalize_budget(cls, raw: str) -> Tuple[str, bool]:
        """
        Normalize budget ranges.
        Returns (normalized_value, was_normalized).
        """
        normalized = cls.BUDGET_NORMALIZATIONS.get(raw.lower(), raw.lower().replace(" ", "_"))
        was_normalized = normalized.lower() != raw.lower()
        return normalized, was_normalized

# =============================================================================
# SECTION 7: EXTRACTION PIPELINE
# =============================================================================

class ExtractionPipeline:
    """
    The core compiler logic: envelopes → canonical packet.
    Extracts facts with full provenance, separates from derived signals and hypotheses.
    """
    
    def __init__(self, model_client=None):
        self.model_client = model_client
    
    def extract(self, envelopes: List[SourceEnvelope]) -> CanonicalPacket:
        """
        Main entry point: takes envelopes, produces canonical packet.
        """
        packet = CanonicalPacket(
            packet_id=f"pkt_{uuid.uuid4().hex[:8]}",
            created_at=datetime.now().isoformat(),
            last_updated=datetime.now().isoformat()
        )
        
        for envelope in envelopes:
            packet.source_envelope_ids.append(envelope.envelope_id)
            
            if envelope.content_type == "freeform_text":
                self._extract_from_freeform(envelope, packet)
            elif envelope.content_type == "structured_json":
                self._extract_from_structured(envelope, packet)
            elif envelope.content_type == "hybrid":
                self._extract_from_hybrid(envelope, packet)
        
        # After extraction, compute derived signals
        self._compute_derived_signals(packet)
        
        # Identify unknowns
        self._identify_unknowns(packet)
        
        return packet
    
    def _extract_from_freeform(self, envelope: SourceEnvelope, packet: CanonicalPacket) -> None:
        """Extract facts from freeform text with proper provenance"""
        text = envelope.content
        
        # In production, this would call LLM. For prototype, we demonstrate the pattern.
        extraction_results = self._mock_llm_extraction(text, envelope.envelope_id)
        
        for field_name, result in extraction_results.items():
            # CRITICAL: Route to correct layer based on authority
            if result["authority_level"] == "soft_hypothesis":
                # This is a hypothesis, not a fact
                slot = Slot(
                    value=result["value"],
                    confidence=result["confidence"],
                    authority_level="soft_hypothesis",
                    extraction_mode="derived",  # Hypotheses are derived
                    evidence_refs=[EvidenceRef(
                        ref_id=f"ref_{uuid.uuid4().hex[:6]}",
                        envelope_id=envelope.envelope_id,
                        evidence_type="text_span",
                        excerpt=result["excerpt"]
                    )]
                )
                packet.set_hypothesis(field_name, slot)
            else:
                # This is an explicit fact
                slot = Slot(
                    value=result["value"],
                    confidence=result["confidence"],
                    authority_level=result["authority_level"],
                    extraction_mode="direct_extract",
                    evidence_refs=[EvidenceRef(
                        ref_id=f"ref_{uuid.uuid4().hex[:6]}",
                        envelope_id=envelope.envelope_id,
                        evidence_type="text_span",
                        excerpt=result["excerpt"],
                        offset=result.get("offset")
                    )]
                )
                packet.set_fact(field_name, slot)
    
    def _extract_from_structured(self, envelope: SourceEnvelope, packet: CanonicalPacket) -> None:
        """Extract facts from structured data with field paths"""
        data = envelope.content
        
        # Map structured fields to canonical fields
        field_mappings = {
            "destination": "destination_city",
            "origin": "origin_city",
            "travelers": "traveler_count",
            "budget": "budget_range",
            "dates": "travel_dates"
        }
        
        for src_field, canonical_field in field_mappings.items():
            if src_field in data:
                value = data[src_field]
                
                # Apply normalization if applicable
                if canonical_field == "origin_city" or canonical_field == "destination_city":
                    value, was_normalized = Normalizer.normalize_city(str(value))
                    extraction_mode = "normalized" if was_normalized else "imported"
                elif canonical_field == "budget_range":
                    value, was_normalized = Normalizer.normalize_budget(str(value))
                    extraction_mode = "normalized" if was_normalized else "imported"
                else:
                    extraction_mode = "imported"
                
                slot = Slot(
                    value=value,
                    confidence=1.0,
                    authority_level="imported_structured",
                    extraction_mode=extraction_mode,
                    evidence_refs=[EvidenceRef(
                        ref_id=f"ref_{uuid.uuid4().hex[:6]}",
                        envelope_id=envelope.envelope_id,
                        evidence_type="structured_field",
                        excerpt=str(value),
                        field_path=src_field
                    )]
                )
                packet.set_fact(canonical_field, slot)
    
    def _extract_from_hybrid(self, envelope: SourceEnvelope, packet: CanonicalPacket) -> None:
        """
        Handle hybrid input - check for contradictions between freeform and structured.
        Uses named policy for conflict resolution.
        """
        text = envelope.content.get("text", "")
        structured = envelope.content.get("structured", {})
        
        # Extract from both
        freeform_results = self._mock_llm_extraction(text, envelope.envelope_id)
        
        # Check for conflicts
        for field_name, freeform_result in freeform_results.items():
            structured_field = self._canonical_to_structured_field(field_name)
            
            if structured_field in structured:
                structured_value = structured[structured_field]
                freeform_value = freeform_result["value"]
                
                # Normalize structured for comparison
                normalized_structured, _ = Normalizer.normalize_city(str(structured_value))
                
                # Conflict detection
                if str(normalized_structured).lower() != str(freeform_value).lower():
                    # Use named policy for resolution
                    resolution = ConflictResolutionPolicy.resolve(
                        field_name,
                        freeform_value, freeform_result["authority_level"], freeform_result["confidence"],
                        structured_value, "imported_structured", 1.0
                    )
                    
                    packet.add_contradiction(
                        field_name,
                        [freeform_value, structured_value],
                        ["freeform_text", "structured_data"]
                    )
                    
                    slot = Slot(
                        value=resolution["resolved_value"],
                        confidence=1.0,
                        authority_level=resolution["authority"],
                        extraction_mode=resolution["extraction_mode"],
                        evidence_refs=[EvidenceRef(
                            ref_id=f"ref_{uuid.uuid4().hex[:6]}",
                            envelope_id=envelope.envelope_id,
                            evidence_type="structured_field",
                            excerpt=str(structured_value),
                            field_path=structured_field
                        )],
                        notes=f"Conflict resolved: {resolution['resolution_reason']}"
                    )
                    packet.set_fact(field_name, slot)
                else:
                    # Agreement - use structured with note
                    slot = Slot(
                        value=structured_value,
                        confidence=1.0,
                        authority_level="imported_structured",
                        extraction_mode="imported",
                        evidence_refs=[EvidenceRef(
                            ref_id=f"ref_{uuid.uuid4().hex[:6]}",
                            envelope_id=envelope.envelope_id,
                            evidence_type="structured_field",
                            excerpt=str(structured_value),
                            field_path=structured_field
                        )],
                        notes="Confirmed by both freeform and structured"
                    )
                    packet.set_fact(field_name, slot)
            else:
                # Only in freeform - route based on authority level
                if freeform_result["authority_level"] == "soft_hypothesis":
                    slot = Slot(
                        value=freeform_result["value"],
                        confidence=freeform_result["confidence"],
                        authority_level="soft_hypothesis",
                        extraction_mode="derived",
                        evidence_refs=[EvidenceRef(
                            ref_id=f"ref_{uuid.uuid4().hex[:6]}",
                            envelope_id=envelope.envelope_id,
                            evidence_type="text_span",
                            excerpt=freeform_result["excerpt"]
                        )]
                    )
                    packet.set_hypothesis(field_name, slot)
                else:
                    slot = Slot(
                        value=freeform_result["value"],
                        confidence=freeform_result["confidence"],
                        authority_level=freeform_result["authority_level"],
                        extraction_mode="direct_extract",
                        evidence_refs=[EvidenceRef(
                            ref_id=f"ref_{uuid.uuid4().hex[:6]}",
                            envelope_id=envelope.envelope_id,
                            evidence_type="text_span",
                            excerpt=freeform_result["excerpt"]
                        )]
                    )
                    packet.set_fact(field_name, slot)
    
    def _compute_derived_signals(self, packet: CanonicalPacket) -> None:
        """
        Compute derived signals from facts.
        CRITICAL: Derived signals are computed from FACTS only, never from hypotheses.
        """
        # Example: If we have destination and origin, we can derive trip_type
        if "destination_city" in packet.facts and "origin_city" in packet.facts:
            dest = packet.facts["destination_city"].value
            origin = packet.facts["origin_city"].value
            
            # International vs domestic
            international_destinations = ["Japan", "Paris", "Tokyo", "London", "New York"]
            is_international = dest in international_destinations and origin not in international_destinations
            
            if is_international:
                slot = Slot(
                    value="international",
                    confidence=1.0,  # Derived with certainty from facts
                    authority_level="derived_signal",
                    extraction_mode="derived",
                    evidence_refs=[
                        EvidenceRef(
                            ref_id=f"ref_{uuid.uuid4().hex[:6]}",
                            envelope_id="derived",
                            evidence_type="derived",
                            excerpt=f"Computed from destination={dest}, origin={origin}"
                        )
                    ],
                    notes="Derived from explicit destination and origin facts"
                )
                packet.set_derived_signal("trip_type", slot)
        
        # Example: If we have traveler_count, derive group_type
        if "traveler_count" in packet.facts:
            count = packet.facts["traveler_count"].value
            if count == 1:
                group_type = "solo"
            elif count == 2:
                group_type = "couple"
            elif count > 2:
                group_type = "group"
            else:
                group_type = "unknown"
            
            if group_type != "unknown":
                slot = Slot(
                    value=group_type,
                    confidence=1.0,
                    authority_level="derived_signal",
                    extraction_mode="derived",
                    evidence_refs=[
                        EvidenceRef(
                            ref_id=f"ref_{uuid.uuid4().hex[:6]}",
                            envelope_id="derived",
                            evidence_type="derived",
                            excerpt=f"Computed from traveler_count={count}"
                        )
                    ],
                    notes="Derived from explicit traveler_count fact"
                )
                packet.set_derived_signal("group_type", slot)
    
    def _mock_llm_extraction(self, text: str, envelope_id: str) -> Dict[str, Any]:
        """
        Mock LLM extraction for prototype.
        In production, this would call the actual model.
        """
        results = {}
        text_lower = text.lower()
        
        # Destination detection (explicit fact)
        if "japan" in text_lower:
            results["destination_city"] = {
                "value": "Japan",
                "confidence": 0.95,
                "excerpt": "trip to Japan",
                "offset": {"start": text_lower.find("japan"), "end": text_lower.find("japan") + 5},
                "authority_level": "explicit_user",
                "is_explicit": True
            }
        
        # Origin detection (explicit fact)
        if "bangalore" in text_lower or "blr" in text_lower:
            results["origin_city"] = {
                "value": "Bangalore",
                "confidence": 0.95,
                "excerpt": "from Bangalore" if "bangalore" in text_lower else "blr",
                "offset": {"start": text_lower.find("bangalore"), "end": text_lower.find("bangalore") + 9},
                "authority_level": "explicit_user",
                "is_explicit": True
            }
        
        # Budget detection (explicit fact)
        if "mid-range" in text_lower or "mid range" in text_lower:
            results["budget_range"] = {
                "value": "mid_range",
                "confidence": 0.85,
                "excerpt": "mid-range" if "mid-range" in text_lower else "mid range",
                "authority_level": "explicit_user",
                "is_explicit": True
            }
        
        # Traveler hints (HYPOTHESIS, not fact)
        # This should go to hypotheses dict, not facts
        if "toddler" in text_lower or "kids" in text_lower:
            results["traveler_profile_hint"] = {
                "value": "family_with_children",
                "confidence": 0.7,
                "excerpt": "toddler" if "toddler" in text_lower else "kids",
                "authority_level": "soft_hypothesis",  # CRITICAL: This is a hypothesis
                "is_explicit": False
            }
        
        return results
    
    def _canonical_to_structured_field(self, canonical: str) -> str:
        """Reverse mapping"""
        reverse_map = {
            "destination_city": "destination",
            "origin_city": "origin",
            "traveler_count": "travelers",
            "budget_range": "budget",
            "travel_dates": "dates"
        }
        return reverse_map.get(canonical, canonical)
    
    def _identify_unknowns(self, packet: CanonicalPacket) -> None:
        """Mark expected fields that are not present"""
        required_fields = [
            "destination_city", "origin_city", "travel_dates",
            "traveler_count", "budget_range"
        ]
        
        for field in required_fields:
            if field not in packet.facts:
                packet.add_unknown(field, "not_present_in_source")

# =============================================================================
# TEST SUITE - Multiple Scenarios
# =============================================================================

print("=" * 80)
print("TEST 1: Freeform Text Extraction (with normalization and hypothesis routing)")
print("=" * 80)

adapter = SourceAdapter()
pipeline = ExtractionPipeline()

# Test 1: Pure freeform with hypothesis
envelope1 = adapter.adapt_freeform(
    "Client wants a nature trip to Japan. Starting from BLR. Budget is mid-range. Traveling with a toddler.",
    source="agency_notes",
    actor="agent"
)
packet1 = pipeline.extract([envelope1])

print(f"\nPacket ID: {packet1.packet_id}")
print(f"Source Envelopes: {packet1.source_envelope_ids}")
print(f"\n=== FACTS (Explicit extracted) ===")
for field, slot in packet1.facts.items():
    print(f"  {field}: {slot.value}")
    print(f"    Authority: {slot.authority_level}, Mode: {slot.extraction_mode}, Confidence: {slot.confidence}")
    print(f"    Evidence: {slot.evidence_refs[0].excerpt}")

print(f"\n=== DERIVED SIGNALS (Computed from facts) ===")
for field, slot in packet1.derived_signals.items():
    print(f"  {field}: {slot.value}")
    print(f"    Authority: {slot.authority_level}")
    print(f"    Evidence: {slot.evidence_refs[0].excerpt}")

print(f"\n=== HYPOTHESES (Weak inferences) ===")
for field, slot in packet1.hypotheses.items():
    print(f"  {field}: {slot.value}")
    print(f"    Authority: {slot.authority_level}, Confidence: {slot.confidence}")
    print(f"    Evidence: {slot.evidence_refs[0].excerpt}")

print(f"\n=== UNKNOWNS (Explicit missing fields) ===")
for u in packet1.unknowns:
    print(f"  {u.field_name}: {u.reason}")

print(f"\nEvent log entries: {len(packet1.events)}")

print("\n" + "=" * 80)
print("TEST 2: Structured Data Extraction (with normalization demonstration)")
print("=" * 80)

envelope2 = adapter.adapt_structured({
    "destination": "Paris",
    "origin": "blr",  # Will be normalized to Bangalore
    "travelers": 2,
    "budget": "mid-range",  # Will be normalized to mid_range
    "dates": "2024-06-15 to 2024-06-22"
}, source="traveler_form", actor="traveler")

packet2 = pipeline.extract([envelope2])

print(f"\n=== FACTS (with extraction_mode='imported' or 'normalized') ===")
for field, slot in packet2.facts.items():
    print(f"  {field}: {slot.value}")
    print(f"    Extraction mode: {slot.extraction_mode}")
    print(f"    Authority: {slot.authority_level}")
    print(f"    Field path: {slot.evidence_refs[0].field_path}")

print(f"\n=== DERIVED SIGNALS ===")
for field, slot in packet2.derived_signals.items():
    print(f"  {field}: {slot.value}")
    print(f"    Evidence: {slot.evidence_refs[0].excerpt}")

print("\n" + "=" * 80)
print("TEST 3: Hybrid Input with Named Conflict Resolution Policy")
print("=" * 80)

envelope3 = adapter.adapt_hybrid(
    text="Starting from Bangalore. Going to Tokyo.",
    data={"origin": "Mumbai", "destination": "Tokyo"},  # Conflict: origin differs
    source="agency_notes",
    actor="agent"
)

packet3 = pipeline.extract([envelope3])

print(f"\n=== FACTS (with conflict resolution) ===")
for field, slot in packet3.facts.items():
    print(f"  {field}: {slot.value}")
    print(f"    Authority: {slot.authority_level}")
    print(f"    Extraction mode: {slot.extraction_mode}")
    if slot.notes:
        print(f"    Notes: {slot.notes}")

print(f"\n=== CONTRADICTIONS ===")
for c in packet3.contradictions:
    print(f"  {c['field_name']}: {c['values']}")
    print(f"    Sources: {c['sources']}")

print("\n" + "=" * 80)
print("TEST 4: Authority Precedence Verification")
print("=" * 80)

print("\nAuthority Level Order (lower number = higher authority):")
for level in AuthorityLevel:
    print(f"  {level.value}: {level.name}")

print("\nAuthority comparison tests:")
print(f"  higher_authority('explicit_user', 'imported_structured') = {higher_authority('explicit_user', 'imported_structured')}")
print(f"  higher_authority('imported_structured', 'explicit_owner') = {higher_authority('imported_structured', 'explicit_owner')}")
print(f"  higher_authority('derived_signal', 'soft_hypothesis') = {higher_authority('derived_signal', 'soft_hypothesis')}")

print("\n" + "=" * 80)
print("TEST 5: Full Packet Export (JSON)")
print("=" * 80)

def packet_to_dict(packet):
    """Convert packet to serializable dict"""
    return {
        "packet_id": packet.packet_id,
        "stage": packet.stage,
        "facts": {k: asdict(v) for k, v in packet.facts.items()},
        "derived_signals": {k: asdict(v) for k, v in packet.derived_signals.items()},
        "hypotheses": {k: asdict(v) for k, v in packet.hypotheses.items()},
        "unknowns": [asdict(u) for u in packet.unknowns],
        "contradictions": packet.contradictions,
        "source_envelope_ids": packet.source_envelope_ids,
        "event_count": len(packet.events)
    }

print(json.dumps(packet_to_dict(packet1), indent=2))